In [1]:
import requests
from dotenv import load_dotenv
import os
from transformers import pipeline

load_dotenv()
api_key = os.getenv("NEWS_API_KEY")

# Modelimizi yükle
classifier = pipeline(
    "text-classification",
    model="Eelis/turkish-financial-bert-v2"
)

print("Model yüklendi.")

config.json:   0%|          | 0.00/893 [00:00<?, ?B/s]

C:\Users\Admin\PycharmProjects\turkish-financial-sentiment\.venv\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--Eelis--turkish-financial-bert-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model yüklendi.


In [2]:
def haberleri_cek(arama_terimi, sayfa_boyutu=10):
    url = "https://newsapi.org/v2/everything"
    params = {
        "q": arama_terimi,
        "language": "tr",
        "pageSize": sayfa_boyutu,
        "sortBy": "publishedAt",
        "apiKey": api_key
    }
    response = requests.get(url, params=params, timeout=10)
    if response.status_code == 200:
        return response.json()["articles"]
    else:
        print(f"Hata: {response.status_code}")
        return []

def analiz_et(haberler):
    sonuclar = []
    for haber in haberler:
        baslik = haber["title"]
        if not baslik or baslik == "[Removed]":
            continue
        tahmin = classifier(baslik)[0]
        sonuclar.append({
            "baslik": baslik,
            "etiket": tahmin["label"],
            "guven": round(tahmin["score"], 2),
            "tarih": haber["publishedAt"][:10],
            "kaynak": haber["source"]["name"]
        })
    return sonuclar

# Canlı test
haberler = haberleri_cek("borsa hisse yatırım")
sonuclar = analiz_et(haberler)

for s in sonuclar:
    emoji = "🟢" if s["etiket"] == "bullish" else "🔴" if s["etiket"] == "bearish" else "⚪"
    print(f"{emoji} [{s['etiket'].upper()}] ({s['guven']}) - {s['tarih']}")
    print(f"   {s['baslik']}")
    print(f"   Kaynak: {s['kaynak']}\n")

⚪ [NEUTRAL] (0.98) - 2026-03-29
   BIST 100’de haftalık kayıp yüzde 2,68: En çok kazandıran ve kaybettiren hisseler belli oldu
   Kaynak: Yenicaggazetesi.com

🔴 [BEARISH] (1.0) - 2026-03-29
   Savaş baskısı sürüyor
   Kaynak: Hurriyet.com.tr

⚪ [NEUTRAL] (1.0) - 2026-03-28
   Türkiye'nin Gayrimenkul devi halka arz oluyor: İşte hisse fiyatları
   Kaynak: Yenicaggazetesi.com

⚪ [NEUTRAL] (1.0) - 2026-03-28
   'Türkiye 60 ton altın sattı' denildi... Savaşın 25 günlük bilançosu ne... Haftanın en çok kazandıran yatırımı: Ne altın ne dolar
   Kaynak: Odatv.com

🟢 [BULLISH] (1.0) - 2026-03-27
   Altın, borsa, euro ve dolar: Yatırımcıya en çok ne kazandırdı?
   Kaynak: Yenicaggazetesi.com

🔴 [BEARISH] (1.0) - 2026-03-27
   Yatırım araçlarında bu hafta değer kaybında altın ilk sırada, ikinci sırada hisse senetleri yer aldı
   Kaynak: Patronlardunyasi.com

⚪ [NEUTRAL] (1.0) - 2026-03-25
   Mert Başaran küçük yatırımcıya çok kazandıracak reçeteyi verdi: Altın, dolar borsa mı?
   Kaynak: Yenicagga